# Convertir materiales de semanas a PDF

Este notebook convierte los archivos `.md` y `.ipynb` de `semana_01`, `semana_02` y `semana_03` a PDF.

- Excluye automaticamente `cuestionario.md`.
- Guarda los PDF en la carpeta `pdf/`, manteniendo la estructura por semana.
- Usa HTML + Chromium/Playwright, por lo que no requiere LaTeX.
- Si falta alguna libreria, la instala en el ambiente activo.

Recomendado: correr este notebook con el kernel `Python (.venv_pdf MA_M3 PDF)`.


In [ ]:
# Instalar/verificar dependencias necesarias
import importlib.util
import subprocess
import sys

paquetes = {
    "nbconvert": "nbconvert",
    "nbformat": "nbformat",
    "nbclient": "nbclient",
    "markdown": "markdown",
    "playwright": "playwright",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}

faltantes = [pip_name for import_name, pip_name in paquetes.items() if importlib.util.find_spec(import_name) is None]

if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
else:
    print("Dependencias Python OK")

# Instala Chromium si Playwright aun no lo tiene disponible.
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        browser = p.chromium.launch()
        browser.close()
    print("Chromium OK")
except Exception:
    print("Instalando Chromium para Playwright...")
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
    print("Chromium instalado")


In [ ]:
from pathlib import Path
from html import escape
from tempfile import TemporaryDirectory
import subprocess
import sys

import markdown
import nbformat
from nbclient import NotebookClient
from nbconvert import HTMLExporter

ROOT = Path.cwd()
SEMANAS = [ROOT / "semana_01", ROOT / "semana_02", ROOT / "semana_03"]
OUTPUT_DIR = ROOT / "pdf"

# Si es True, ejecuta los notebooks antes de exportarlos.
# Si algun notebook depende de internet o demora mucho, puedes dejarlo en False.
EXECUTE_NOTEBOOKS = False
TIMEOUT = 180

CSS = """
body {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    font-size: 11pt;
    line-height: 1.45;
    color: #222;
    max-width: 980px;
    margin: 0 auto;
}
h1, h2, h3 { color: #111; }
pre, code {
    font-family: 'SFMono-Regular', Consolas, monospace;
    font-size: 9pt;
}
pre {
    background: #f5f5f5;
    border: 1px solid #ddd;
    border-radius: 4px;
    padding: 10px;
    overflow-wrap: break-word;
    white-space: pre-wrap;
}
table {
    border-collapse: collapse;
    width: 100%;
    margin: 1em 0;
}
th, td {
    border: 1px solid #ddd;
    padding: 6px 8px;
}
th { background: #f0f0f0; }
img { max-width: 100%; height: auto; }
blockquote {
    border-left: 4px solid #aaa;
    margin-left: 0;
    padding-left: 12px;
    color: #444;
}
@page { size: A4; margin: 1.4cm; }
"""

def html_document(title, body):
    return f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>{escape(title)}</title>
  <style>{CSS}</style>
</head>
<body>{body}</body>
</html>"""

def output_path_for(path):
    rel = path.relative_to(ROOT)
    return (OUTPUT_DIR / rel).with_suffix(".pdf")

def html_to_pdf(html, destino):
    destino.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory() as tmp:
        html_path = Path(tmp) / "documento.html"
        html_path.write_text(html, encoding="utf-8")
        script = """
from pathlib import Path
import sys
from playwright.sync_api import sync_playwright
html_path = Path(sys.argv[1]).resolve()
destino = Path(sys.argv[2]).resolve()
with sync_playwright() as p:
    browser = p.chromium.launch()
    page = browser.new_page()
    page.goto(html_path.as_uri(), wait_until='networkidle')
    page.pdf(
        path=str(destino),
        format='A4',
        print_background=True,
        margin={'top': '14mm', 'right': '14mm', 'bottom': '14mm', 'left': '14mm'},
    )
    browser.close()
"""
        subprocess.check_call([sys.executable, "-c", script, str(html_path), str(destino)])


In [ ]:
def convertir_markdown(path):
    texto = path.read_text(encoding="utf-8")
    body = markdown.markdown(
        texto,
        extensions=["extra", "tables", "fenced_code", "toc"],
        output_format="html5",
    )
    html = html_document(path.name, body)
    destino = output_path_for(path)
    html_to_pdf(html, destino)
    return destino

def convertir_notebook(path):
    nb = nbformat.read(path, as_version=4)

    if EXECUTE_NOTEBOOKS:
        cliente = NotebookClient(nb, timeout=TIMEOUT, kernel_name="python3", resources={"metadata": {"path": str(path.parent)}})
        cliente.execute()

    exporter = HTMLExporter()
    exporter.exclude_input_prompt = True
    exporter.exclude_output_prompt = True
    body, _ = exporter.from_notebook_node(nb)
    html = html_document(path.name, body)

    destino = output_path_for(path)
    html_to_pdf(html, destino)
    return destino


In [ ]:
archivos = []

for semana in SEMANAS:
    archivos.extend(sorted(semana.glob("*.md")))
    archivos.extend(sorted(semana.glob("*.ipynb")))

archivos = [p for p in archivos if p.name != "cuestionario.md"]

print("Archivos a convertir:")
for p in archivos:
    print("-", p.relative_to(ROOT))

In [ ]:
convertidos = []
errores = []

for path in archivos:
    try:
        if path.suffix == ".md":
            destino = convertir_markdown(path)
        elif path.suffix == ".ipynb":
            destino = convertir_notebook(path)
        else:
            continue

        convertidos.append(destino)
        print(f"OK  {path.relative_to(ROOT)} -> {destino.relative_to(ROOT)}")
    except Exception as exc:
        errores.append((path, exc))
        print(f"ERROR  {path.relative_to(ROOT)}: {exc}")

print("\nResumen")
print(f"Convertidos: {len(convertidos)}")
print(f"Errores: {len(errores)}")

if errores:
    print("\nArchivos con error:")
    for path, exc in errores:
        print(f"- {path.relative_to(ROOT)}: {exc}")

## Notas

Para ejecutar desde terminal con el ambiente creado:

```bash
source .venv_pdf/bin/activate
jupyter notebook notebook_to_pdf.ipynb
```

Tambien puedes abrir el notebook desde Jupyter y elegir el kernel `Python (.venv_pdf MA_M3 PDF)`.
